# House Price Prediction — Notebook (Windows-friendly)
This notebook contains a self-contained, interactive training pipeline for the project. It has been cleaned of Colab-specific code and uses local paths.
Set `DATA_PATH` to your CSV file if different and run cells sequentially.

## Setup & Imports
Run this cell to import required packages. If a package is missing, install it in your environment (see `requirements.txt`).

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None

print('Imports OK')

Imports OK


## Helper functions
Data loading, preprocessing and evaluation helper functions used by the notebook.

In [2]:
def load_data(path, sample_frac=None, nrows=None):
    """Load CSV data with optional sampling."""
    if sample_frac is not None and sample_frac < 1.0:
        df = pd.read_csv(path, nrows=nrows)
        return df.sample(frac=sample_frac, random_state=42)
    return pd.read_csv(path, nrows=nrows)


def build_preprocessor(df, target_col):
    """Build sklearn preprocessing pipeline for numeric and categorical features."""
    X = df.drop(columns=[target_col])
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    
    print(f"\nFeature types:")
    print(f"  Numeric: {len(numeric_cols)} columns")
    print(f"  Categorical: {len(categorical_cols)} columns")

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols),
    ])

    return preprocessor


def evaluate_and_report(model, X_test, y_test):
    """Evaluate model and return metrics."""
    preds = model.predict(X_test)
    # use np.sqrt of MSE for compatibility with installed scikit-learn
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    return {'rmse': rmse, 'r2': r2, 'preds': preds}


## Configuration and Data Preview
Set `DATA_PATH` below and run the cell to preview the dataset and confirm the target column name.

In [3]:
# Update this path if your dataset is stored elsewhere
DATA_PATH = Path(r'D:\Do An Tot Nghiep - Du doan gia bat dong san bang ML va DL\Data\cleaned_real_estate.csv')
TARGET = 'price'  # change if needed
PREVIEW_NROWS = 1000

print(f"Loading data preview from: {DATA_PATH}")
df_preview = load_data(DATA_PATH, nrows=PREVIEW_NROWS)
print(f'✓ Loaded {len(df_preview)} rows for preview')
print(f'\nFirst 5 rows:')
display(df_preview.head())

print(f'\nDataset info:')
print(f'  Shape: {df_preview.shape}')
print(f'  Columns: {list(df_preview.columns)}')
print(f'  Target present: {TARGET in df_preview.columns}')

if TARGET in df_preview.columns:
    print(f'\nTarget column "{TARGET}" statistics:')
    print(f'  Count: {df_preview[TARGET].count()}')
    print(f'  Mean:  {df_preview[TARGET].mean():,.2f}')
    print(f'  Min:   {df_preview[TARGET].min():,.2f}')
    print(f'  Max:   {df_preview[TARGET].max():,.2f}')
    print(f'  Missing: {df_preview[TARGET].isna().sum()}')


Loading data preview from: D:\Do An Tot Nghiep - Du doan gia bat dong san bang ML va DL\Data\cleaned_real_estate.csv
✓ Loaded 1000 rows for preview

First 5 rows:


,country,city,date,price,area_m2,property_type,price_per_m2,year,month
0,Singapore,ANG MO KIO,2012-03-01,250000.0,45.0,2 ROOM,5555.555556,2012,3
1,Singapore,ANG MO KIO,2012-03-01,265000.0,44.0,2 ROOM,6022.727273,2012,3
2,Singapore,ANG MO KIO,2012-03-01,315000.0,68.0,3 ROOM,4632.352941,2012,3
3,Singapore,ANG MO KIO,2012-03-01,320000.0,67.0,3 ROOM,4776.119403,2012,3
4,Singapore,ANG MO KIO,2012-03-01,321000.0,67.0,3 ROOM,4791.044776,2012,3



Dataset info:
  Shape: (1000, 9)
  Columns: ['country', 'city', 'date', 'price', 'area_m2', 'property_type', 'price_per_m2', 'year', 'month']
  Target present: True

Target column "price" statistics:
  Count: 1000
  Mean:  453,942.91
  Min:   249,000.00
  Max:   900,000.00
  Missing: 0


## Train (quick run)
This cell trains baseline models on a sample fraction of the data for quick iteration. Adjust `SAMPLE_FRAC` or `NROWS` as needed.

In [4]:
# Training params
SAMPLE_FRAC = 0.2  # Use 20% of data for quick testing (set to 1.0 for full training)
NROWS = None  # Limit rows (None = all rows)
OUT_DIR = Path('models')
OUT_DIR.mkdir(exist_ok=True)
TEST_SIZE = 0.2
RANDOM_STATE = 42

print(f"Loading data (sample_frac={SAMPLE_FRAC})...")
df = load_data(DATA_PATH, sample_frac=SAMPLE_FRAC if SAMPLE_FRAC < 1.0 else None, nrows=NROWS)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

if TARGET not in df.columns:
    available = ', '.join(df.columns[:10])
    raise ValueError(f'Target column "{TARGET}" not found.\nAvailable: {available}')

# Clean data
initial_rows = len(df)
df = df.dropna(subset=[TARGET])
if len(df) < initial_rows:
    print(f"Dropped {initial_rows - len(df)} rows with missing target")

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"\nTarget statistics:")
print(f"  Mean: {y.mean():,.2f}")
print(f"  Min:  {y.min():,.2f}")
print(f"  Max:  {y.max():,.2f}")

# Build preprocessor
preprocessor = build_preprocessor(df, TARGET)

# Configure models with optimized hyperparameters
models = []
models.append(('LinearRegression', Pipeline([('pre', preprocessor), ('est', LinearRegression())])))
models.append(('RidgeCV', Pipeline([('pre', preprocessor), ('est', RidgeCV(alphas=[0.1, 1.0, 10.0], cv=3))])))
models.append(('LassoCV', Pipeline([('pre', preprocessor), ('est', LassoCV(cv=3, max_iter=2000, random_state=RANDOM_STATE))])))
models.append(('RandomForest', Pipeline([('pre', preprocessor), ('est', RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
))])))

if XGBRegressor is not None:
    models.append(('XGBoost', Pipeline([('pre', preprocessor), ('est', XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0
    ))])))
else:
    print('⚠ xgboost not installed — skipping XGBoost')

if LGBMRegressor is not None:
    models.append(('LightGBM', Pipeline([('pre', preprocessor), ('est', LGBMRegressor(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        verbose=-1
    ))])))
else:
    print('⚠ lightgbm not installed — skipping LightGBM')

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"\nTrain/Test split: {len(X_train)} / {len(X_test)} samples")

# Train models
print(f"\nTraining {len(models)} models...")
results = []
import time

for idx, (name, pipe) in enumerate(models, 1):
    print(f"\n[{idx}/{len(models)}] Training {name}...")
    start_time = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    ev = evaluate_and_report(pipe, X_test, y_test)
    results.append({
        'name': name,
        'rmse': ev['rmse'],
        'r2': ev['r2'],
        'train_time': train_time
    })
    
    # Save model
    model_path = OUT_DIR / f'{name}.joblib'
    joblib.dump(pipe, model_path)
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f'  RMSE: {ev["rmse"]:,.2f}, R²: {ev["r2"]:.4f}')
    print(f'  Saved to {model_path} ({size_mb:.2f} MB, {train_time:.1f}s)')

# Sort and display results
results = sorted(results, key=lambda r: r['rmse'])
print('\n' + '='*60)
print('Model summary (best first):')
print('='*60)
for r in results:
    print(f"  {r['name']:20s} RMSE={r['rmse']:>10,.2f}  R²={r['r2']:.4f}  Time={r['train_time']:.1f}s")
print('='*60)


Loading data (sample_frac=0.2)...
Loaded 195754 rows, 9 columns

Target statistics:
  Mean: 336,061.47
  Min:  33,000.00
  Max:  1,085,000.00

Feature types:
  Numeric: 4 columns
  Categorical: 4 columns

Train/Test split: 156603 / 39151 samples

Training 6 models...

[1/6] Training LinearRegression...
Loaded 195754 rows, 9 columns

Target statistics:
  Mean: 336,061.47
  Min:  33,000.00
  Max:  1,085,000.00

Feature types:
  Numeric: 4 columns
  Categorical: 4 columns

Train/Test split: 156603 / 39151 samples

Training 6 models...

[1/6] Training LinearRegression...
  RMSE: 36,697.85, R²: 0.9584
  Saved to models\LinearRegression.joblib (0.03 MB, 22.4s)

[2/6] Training RidgeCV...
  RMSE: 36,697.85, R²: 0.9584
  Saved to models\LinearRegression.joblib (0.03 MB, 22.4s)

[2/6] Training RidgeCV...
  RMSE: 36,636.31, R²: 0.9586
  Saved to models\RidgeCV.joblib (0.02 MB, 26.0s)

[3/6] Training LassoCV...
  RMSE: 36,636.31, R²: 0.9586
  Saved to models\RidgeCV.joblib (0.02 MB, 26.0s)

[3/6] 

d:\Do An Tot Nghiep - Du doan gia bat dong san bang ML va DL\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [5]:
# Save best model (feature-name-safe format)
if 'results' in globals() and len(results):
    best_name = results[0]['name']
    best_candidate = OUT_DIR / 'best.joblib'
    
    print(f"\n{'='*60}")
    print(f"Saving best model: {best_name}")
    print(f"{'='*60}")
    
    try:
        # Load the previously saved model file and re-save with feature names
        m = joblib.load(OUT_DIR / f'{best_name}.joblib')
        feature_names = list(X.columns)
        
        # Save as dict with pipeline and feature names for robust prediction
        joblib.dump({
            'pipeline': m,
            'feature_names': feature_names,
            'model_name': best_name,
            'target': TARGET,
            'rmse': results[0]['rmse'],
            'r2': results[0]['r2']
        }, best_candidate)
        
        print(f"✓ Saved best model to: {best_candidate}")
        print(f"  Model: {best_name}")
        print(f"  RMSE: {results[0]['rmse']:,.2f}")
        print(f"  R²: {results[0]['r2']:.4f}")
        print(f"  Features: {len(feature_names)}")
    except Exception as e:
        print(f'❌ Could not save best model: {e}')
else:
    print('⚠ No models trained yet')



Saving best model: RandomForest
✓ Saved best model to: models\best.joblib
  Model: RandomForest
  RMSE: 2,006.79
  R²: 0.9999
  Features: 8
✓ Saved best model to: models\best.joblib
  Model: RandomForest
  RMSE: 2,006.79
  R²: 0.9999
  Features: 8


## Feature importance (RandomForest)
If `RandomForest` was trained, this cell will attempt to display the top feature importances.

In [6]:
rf_path = OUT_DIR / 'RandomForest.joblib'
if rf_path.exists():
    rf = joblib.load(rf_path)
    try:
        pre = rf.named_steps['pre']
        num_names = pre.transformers_[0][2]
        cat_transformer = pre.transformers_[1][1].named_steps['onehot']
        cat_names = []
        if hasattr(cat_transformer, 'categories_'):
            for col, cats in zip(pre.transformers_[1][2], cat_transformer.categories_):
                cat_names.extend([f'{col}={c}' for c in cats])
        feat_names = list(num_names) + cat_names
        importances = rf.named_steps['est'].feature_importances_
        imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
        display(imp_df.sort_values('importance', ascending=False).head(30))
    except Exception as e:
        print('Could not compute feature names automatically:', e)
else:
    print('RandomForest model not found at', rf_path)

,feature,importance
1,price_per_m2,7.018155e-01
0,area_m2,2.979905e-01
1022,property_type=2 ROOM,8.079293e-05
1023,property_type=3 ROOM,3.150333e-05
2,year,2.462267e-05
924,date=2017-11-02,8.049533e-06
110,city=Brighton East,5.991877e-06
1024,property_type=4 ROOM,5.534591e-06
3,month,4.887247e-06
1025,property_type=5 ROOM,3.664396e-06


## Quick inference example
Load the best saved model and predict on the first rows of the dataset.

In [7]:
best_model = None
if 'results' in globals() and len(results):
    best_name = results[0]['name']
    candidate = OUT_DIR / f'{best_name}.joblib'
    if candidate.exists():
        best_model = joblib.load(candidate)

if best_model is not None:
    sample_X = X.head(10)
    preds = best_model.predict(sample_X)
    display(pd.DataFrame({'prediction': preds}))
else:
    print('No saved model available for quick inference.')

,prediction
0,117788.043674
1,351812.529068
2,237514.345300
3,134951.615606
4,322322.528807
5,43326.412850
6,251433.795242
7,694147.424701
8,132521.349318
9,179899.244235


## Next steps
- Add hyperparameter tuning, SHAP explanations, or a lightweight CLI wrapper.
- See `TrainPipeline_from_py.ipynb` for an alternative notebook converted from the scripts.